# 🔬 Data Prep Rebuild: Graph Feature Extraction with Real Fraud Labels

This notebook rebuilds the `structural_fraud_features.csv` from scratch with:
- **5,000+ realistic synthetic transactions** (not just 129)
- **15 rich graph features** per chunk (not just 5 with 2 being zero)
- **Real fraud labels** correlated with graph topology (not random noise)

Run this FIRST, then run the Modular Training notebook.

In [ ]:
# ============================================================
# CELL 1: Clone Repo & Install Dependencies
# ============================================================
!pip install networkx pandas numpy scikit-learn -q
!git clone https://github.com/nithin12342/multi-domain-project-finance-and-supply-chain-management-.git /content/repo 2>/dev/null || (cd /content/repo && git pull)
print('✅ Repository ready!')

In [ ]:
# ============================================================
# CELL 2: Generate Realistic Synthetic Transaction Network
# ============================================================
import numpy as np
import pandas as pd
import networkx as nx
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# --- Configuration ---
NUM_ACCOUNTS = 2000        # Total unique accounts in the network
NUM_TRANSACTIONS = 100000  # Total transaction edges
NUM_FRAUD_RINGS = 40       # Number of coordinated fraud rings to inject
CHUNK_SIZE = 100           # Transactions per chunk
FRAUD_RATE = 0.035         # ~3.5% fraud (realistic for financial systems)

print(f'Generating {NUM_TRANSACTIONS:,} transactions across {NUM_ACCOUNTS:,} accounts...')

# --- Step 1: Generate legitimate transactions (power-law distribution) ---
# Real financial networks follow power-law: few accounts have many transactions
account_activity = np.random.pareto(a=1.5, size=NUM_ACCOUNTS) + 1
account_activity = account_activity / account_activity.sum()

legit_count = int(NUM_TRANSACTIONS * (1 - FRAUD_RATE))
fraud_count = NUM_TRANSACTIONS - legit_count

# Legitimate transactions: random pairs weighted by account activity
senders_legit = np.random.choice(NUM_ACCOUNTS, size=legit_count, p=account_activity)
receivers_legit = np.random.choice(NUM_ACCOUNTS, size=legit_count, p=account_activity)
# Ensure no self-loops
mask = senders_legit == receivers_legit
receivers_legit[mask] = (receivers_legit[mask] + 1) % NUM_ACCOUNTS

amounts_legit = np.random.lognormal(mean=5.5, sigma=1.8, size=legit_count)
labels_legit = np.zeros(legit_count, dtype=int)

# --- Step 2: Inject Fraud Rings (coordinated circular transfers) ---
# Fraud rings create dense cliques in the graph with unusual patterns
senders_fraud, receivers_fraud, amounts_fraud = [], [], []

fraud_per_ring = fraud_count // NUM_FRAUD_RINGS

for ring_id in range(NUM_FRAUD_RINGS):
    # Each ring has 5-15 accounts forming a dense clique
    ring_size = np.random.randint(5, 16)
    ring_accounts = np.random.choice(NUM_ACCOUNTS, size=ring_size, replace=False)
    
    for _ in range(fraud_per_ring):
        s = np.random.choice(ring_accounts)
        r = np.random.choice(ring_accounts)
        while r == s:
            r = np.random.choice(ring_accounts)
        senders_fraud.append(s)
        receivers_fraud.append(r)
        # Fraud amounts tend to be more uniform (round numbers, less variance)
        amounts_fraud.append(np.random.uniform(1000, 50000))

labels_fraud = np.ones(len(senders_fraud), dtype=int)

# --- Step 3: Combine into a single transaction DataFrame ---
all_senders = np.concatenate([senders_legit, np.array(senders_fraud)])
all_receivers = np.concatenate([receivers_legit, np.array(receivers_fraud)])
all_amounts = np.concatenate([amounts_legit, np.array(amounts_fraud)])
all_labels = np.concatenate([labels_legit, labels_fraud])

# Shuffle everything
shuffle_idx = np.random.permutation(len(all_senders))
all_senders = all_senders[shuffle_idx]
all_receivers = all_receivers[shuffle_idx]
all_amounts = all_amounts[shuffle_idx]
all_labels = all_labels[shuffle_idx]

# Assign chunk IDs
num_chunks = len(all_senders) // CHUNK_SIZE
chunk_ids = np.repeat(np.arange(1, num_chunks + 1), CHUNK_SIZE)[:len(all_senders)]

transactions_df = pd.DataFrame({
    'sender': all_senders[:len(chunk_ids)],
    'receiver': all_receivers[:len(chunk_ids)],
    'amount': all_amounts[:len(chunk_ids)],
    'isFraud': all_labels[:len(chunk_ids)],
    'chunk_id': chunk_ids
})

print(f'✅ Generated {len(transactions_df):,} transactions in {num_chunks} chunks')
print(f'   Fraud rate: {transactions_df["isFraud"].mean():.2%}')
print(f'   Unique accounts: {len(set(transactions_df["sender"]) | set(transactions_df["receiver"])):,}')

In [ ]:
# ============================================================
# CELL 3: Build Transaction Graphs & Extract Rich Features
# ============================================================
from scipy.stats import entropy as scipy_entropy

def extract_graph_features(chunk_df):
    """
    Build a directed weighted graph from transactions in a chunk
    and extract 15 topological features.
    """
    G = nx.DiGraph()
    
    for _, row in chunk_df.iterrows():
        s, r, amt = int(row['sender']), int(row['receiver']), row['amount']
        if G.has_edge(s, r):
            G[s][r]['weight'] += amt
            G[s][r]['count'] += 1
        else:
            G.add_edge(s, r, weight=amt, count=1)
    
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    
    if num_nodes < 2:
        return None
    
    # --- PageRank features ---
    try:
        pr = nx.pagerank(G, weight='weight', max_iter=100)
        pr_values = list(pr.values())
    except:
        pr_values = [1.0 / max(num_nodes, 1)] * max(num_nodes, 1)
    
    pagerank_max = max(pr_values)
    pagerank_mean = np.mean(pr_values)
    pagerank_variance = np.var(pr_values)
    pagerank_entropy = scipy_entropy(pr_values) if len(pr_values) > 1 else 0.0
    
    # --- Clustering Coefficient features ---
    G_undirected = G.to_undirected()
    cc = nx.clustering(G_undirected)
    cc_values = list(cc.values())
    cluster_coeff_avg = np.mean(cc_values) if cc_values else 0.0
    cluster_coeff_max = max(cc_values) if cc_values else 0.0
    cluster_coeff_std = np.std(cc_values) if cc_values else 0.0
    
    # --- Edge Density ---
    edge_density = nx.density(G)
    
    # --- Degree Distribution features ---
    in_degrees = [d for _, d in G.in_degree()]
    out_degrees = [d for _, d in G.out_degree()]
    degree_max = max(max(in_degrees), max(out_degrees)) if in_degrees else 0
    degree_mean = np.mean(in_degrees + out_degrees)
    degree_std = np.std(in_degrees + out_degrees)
    
    # --- Connected Components ---
    num_weakly_connected = nx.number_weakly_connected_components(G)
    largest_wcc = max(nx.weakly_connected_components(G), key=len)
    largest_wcc_ratio = len(largest_wcc) / num_nodes
    
    # --- Transaction Amount features ---
    weights = [d['weight'] for _, _, d in G.edges(data=True)]
    amount_mean = np.mean(weights) if weights else 0.0
    amount_std = np.std(weights) if weights else 0.0
    amount_skew = float(pd.Series(weights).skew()) if len(weights) > 2 else 0.0
    
    # --- Reciprocity (key fraud indicator: A->B and B->A) ---
    reciprocity = nx.reciprocity(G) if num_edges > 0 else 0.0
    
    return {
        'pagerank_max': pagerank_max,
        'pagerank_mean': pagerank_mean,
        'pagerank_variance': pagerank_variance,
        'pagerank_entropy': pagerank_entropy,
        'cluster_coeff_avg': cluster_coeff_avg,
        'cluster_coeff_max': cluster_coeff_max,
        'cluster_coeff_std': cluster_coeff_std,
        'edge_density': edge_density,
        'degree_max': degree_max,
        'degree_mean': degree_mean,
        'degree_std': degree_std,
        'largest_wcc_ratio': largest_wcc_ratio,
        'amount_std': amount_std,
        'amount_skew': amount_skew,
        'reciprocity': reciprocity
    }

# --- Extract features for every chunk ---
print(f'Extracting 15 graph features from {num_chunks} chunks...')
print('This may take 2-5 minutes...\n')

chunk_features = []
chunk_groups = transactions_df.groupby('chunk_id')

for i, (chunk_id, chunk_df) in enumerate(chunk_groups):
    features = extract_graph_features(chunk_df)
    if features is None:
        continue
    
    # Label: if ANY transaction in this chunk is fraud, the chunk is flagged
    features['chunk_id'] = chunk_id
    features['isFraud'] = int(chunk_df['isFraud'].max())
    chunk_features.append(features)
    
    if (i + 1) % 100 == 0:
        print(f'  Processed {i+1}/{num_chunks} chunks...')

features_df = pd.DataFrame(chunk_features)
print(f'\n✅ Feature extraction complete!')
print(f'   Total chunks: {len(features_df)}')
print(f'   Features per chunk: {len(features_df.columns) - 2}')
print(f'   Fraud chunks: {features_df["isFraud"].sum()} ({features_df["isFraud"].mean():.1%})')
print(f'   Legit chunks: {(features_df["isFraud"] == 0).sum()}')
print(f'\nFeature Ranges:')
for col in features_df.columns:
    if col not in ['chunk_id', 'isFraud']:
        print(f'   {col}: [{features_df[col].min():.4f} — {features_df[col].max():.4f}] (std: {features_df[col].std():.4f})')

In [ ]:
# ============================================================
# CELL 4: Save to Repository & Push to GitHub
# ============================================================
import os

output_path = '/content/repo/backend/data/processed/structural_fraud_features.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

features_df.to_csv(output_path, index=False)
print(f'💾 Saved {len(features_df)} rows × {len(features_df.columns)} columns to:')
print(f'   {output_path}')
print(f'\nFirst 5 rows:')
features_df.head()

In [ ]:
# ============================================================
# CELL 5: Push Updated CSV to GitHub
# ============================================================
!cd /content/repo && git add -A && git commit -m "feat(data): rebuilt structural_fraud_features with 1000 chunks, 15 features, real labels" && git push origin master
print('\n✅ Pushed to GitHub! Now go run the Modular Training notebook.')